# Pelajaran 13 - Memori Ejen


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-5-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Jenis Memori Ejen

Ejen AI boleh menggunakan pelbagai jenis memori, setiap satu berfungsi untuk tujuan yang berbeza:

### Memori Kerja
Urutan perbualan itu sendiri — mesej yang dipertukarkan dalam satu sesi. Ejen boleh merujuk kembali kepada mesej-mesej terdahulu dalam urutan yang sama untuk mengekalkan koheren. Dalam MAF ini dicipta melalui **`agent.create_session()`**, yang mengembalikan `AgentSession`.

### Memori Jangka Pendek
Maklumat yang kekal untuk tempoh tugasan atau sesi tetapi tidak disimpan secara kekal. Sebagai contoh, ejen mungkin mengumpul fakta semasa perbualan perancangan berbilang giliran dan menggunakannya untuk menghasilkan jadual akhir.

### Memori Jangka Panjang
Keutamaan dan fakta yang kekal **merentas sesi**. Pengguna yang kembali tidak perlu mengulangi sekatan diet atau gaya perjalanan mereka. Memori jangka panjang biasanya disokong oleh storan luaran — pangkalan data, fail, atau indeks vektor — dan dipaparkan kepada ejen melalui alat.


## Memori Kerja dengan Sesi

Bentuk memori yang paling mudah ialah sesi perbualan. Apabila anda menyerahkan sesi yang sama (dibuat melalui `agent.create_session()`) kepada panggilan `agent.run()` berturut-turut, ejen dapat melihat keseluruhan sejarah perbualan itu dan boleh mengingati butiran sebelumnya.

Mari buat ejen pelancongan dan demo memori kerja.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Ejen mengingati belanjawan dengan betul kerana kedua-dua mesej berkongsi sesi yang sama. Ini adalah **memori kerja** — ia wujud hanya untuk jangka hayat sesi tersebut.

### Apa yang berlaku dengan benang baru?

Jika kita membuat sesi **baru**, ejen tidak mempunyai memori mengenai perbualan sebelumnya:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Corak Memori Jangka Panjang

Untuk mengingati keutamaan pengguna **merentas sesi**, kita memerlukan stor berterusan yang hidup di luar benang perbualan. Ejen mengakses stor ini melalui **alatan** — fungsi yang boleh dipanggil untuk menyimpan dan mendapatkan maklumat.

Di bawah kami melaksanakan stor keutamaan ringkas dalam memori (dalam produksi anda akan menyokong ini dengan pangkalan data atau indeks vektor) dan mendedahkannya sebagai alatan yang boleh digunakan ejen.

### Seni Bina
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Senario 1 — Pengguna kali pertama menempah perjalanan ulang tahun

Sarah melawat buat kali pertama. Ejen harus menyimpan keutamaan beliau melalui alat dan menggunakannya untuk mengesyorkan hotel.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Senario 2 — Sarah kembali beberapa minggu kemudian

Sarah memulakan **rantaian baru** (mensimulasikan sesi baru). Ingatan kerja adalah kosong, tetapi stor keutamaan jangka panjang masih mempunyai maklumatnya. Ejen harus mengambilnya dan menggunakannya untuk memperibadikan cadangan.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Ringkasan

Dalam pelajaran ini anda telah mempelajari tiga jenis memori ejen dan cara mengimplementasikannya dengan Microsoft Agent Framework:

| Jenis Memori | Mekanisme MAF | Tempoh Hayat |
|---|---|---|
| **Kerja** | `agent.create_session()` | Perbualan tunggal |
| **Jangka pendek** | Konteks terkumpul dalam satu utas | Tugasan / sesi tunggal |
| **Jangka panjang** | Simpanan luaran diakses melalui fungsi `@tool` | Merentas sesi |

### Pengajaran Utama
1. **`agent.create_session()`** menyediakan memori kerja — ejen dapat melihat seluruh sejarah perbualan dalam satu sesi.
2. **Sesi baru kehilangan konteks** — tanpa memori jangka panjang, ejen tidak dapat mengingati perbualan lalu.
3. **Fungsi `@tool`** merapatkan jurang — mereka membolehkan ejen menyimpan dan mengambil maklumat dari storan kekal.
4. **Personalisasi bertambah baik dari masa ke masa** — lebih banyak keutamaan disimpan, lebih baik cadangan ejen.

### Aplikasi Dunia Sebenar
- **Perkhidmatan Pelanggan**: Mengingati sejarah dan keutamaan pelanggan
- **Pembantu Peribadi**: Mengekalkan konteks merentasi hari atau minggu
- **Penjagaan Kesihatan**: Menjejak maklumat dan keutamaan pesakit
- **E-dagang**: Membeli-belah yang diperibadikan berdasarkan sejarah

### Langkah Seterusnya
- Gantikan dict dalam memori dengan pangkalan data atau storan vektor (contohnya Azure AI Search)
- Tambah luput memori untuk maklumat sensitif masa
- Bina sistem berbilang ejen dengan memori berkongsi
- Terokai buku nota Cognee untuk memori berasaskan graf pengetahuan


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
